# A股原始数据下载

本 Notebook 下载四份必需的 adata 源缓存，并在尾部提供两份附录 A 可选数据的断点下载入口；不执行清洗、复权乘数、VWAP、行业中性化、市值或 Barra 因子计算：

- `adata_listing_dates.parquet`：`all_code()` 股票主表；
- `market_data.parquet`：`k_type=1, adjust_type=2` 后复权 OHLCVA；
- `raw_close.parquet`：`k_type=1, adjust_type=0` 不复权收盘价；
- `stock_shares_history.parquet`：`get_stock_shares(is_history=True)` 历史股本变更记录。

尾部可选数据为核心财务指标与历史分红记录。它们当前不进入第一版 Barra 惩罚，只为附录 A 后续研究预先保存原始输入。长任务均由你手动运行；下载中断后，保持 `force_update=False` 并重新运行对应单元即可从分段缓存继续。

## 1. 首次环境安装（在 PowerShell 终端执行，不在 Notebook 内执行）

```powershell
$project = 'D:\实习\Gflownet因子挖掘'
& 'D:\Miniconda3\Scripts\conda.exe' create -p "$project\.venv" python=3.12 -y
& "$project\.venv\python.exe" -m pip install --upgrade pip
& "$project\.venv\python.exe" -m pip install -r "$project\requirements.txt"
& "$project\.venv\python.exe" -m ipykernel install --user --name gflownet-factor --display-name 'Python (GFlowNet Factor)'
```

安装后将 Notebook 内核切换为 `Python (GFlowNet Factor)`。

In [ ]:
# 2. 环境与路径
import sys
from pathlib import Path

project_root = Path.cwd().resolve()
if project_root.name.lower() == 'notebooks':
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import adata
from factor_gfn.data.downloader import (
    DEFAULT_START_DATE,
    LISTING_DATES_PATH,
    MARKET_DATA_PATH,
    RAW_CLOSE_PATH,
    STOCK_SHARES_PATH,
    download_adjusted_market,
    download_raw_close,
    download_stock_shares,
    download_stock_list,
    print_download_summary,
)

print('Python:', sys.version)
print('adata:', getattr(adata, '__version__', 'unknown'))
print('项目目录:', project_root)
print('开始日期:', DEFAULT_START_DATE)
print('股票主表:', LISTING_DATES_PATH)
print('后复权行情:', MARKET_DATA_PATH)
print('不复权收盘价:', RAW_CLOSE_PATH)
print('历史股本:', STOCK_SHARES_PATH)

In [ ]:
# 3. 下载 all_code() 股票主表（通常很快）
# 已有有效缓存时直接读取；确需刷新时改为 force_update=True。
stocks = download_stock_list(force_update=False)
display(stocks.head())
print(stocks['exchange'].value_counts(dropna=False))

In [ ]:
# 4. 断点下载后复权日行情：k_type=1, adjust_type=2
# end_date=None 表示运行当天。中断后原样重跑本单元即可续传。
adjusted_result = download_adjusted_market(
    start_date='2010-01-01',
    end_date=None,
    force_update=False,
)
adjusted_result

In [ ]:
# 5. 断点下载不复权收盘价：k_type=1, adjust_type=0
# 只保存 trade_date、stock_code、close。
raw_close_result = download_raw_close(
    start_date='2010-01-01',
    end_date=None,
    force_update=False,
)
raw_close_result

In [ ]:
# 6. 断点下载历史股本：get_stock_shares(is_history=True)
# 保存总股本、限售股本、流通A股股本和变更原因；中断后原样重跑即可续传。

from factor_gfn.data.downloader import (

    download_stock_shares,
)



shares_result = download_stock_shares(force_update=False)
shares_result

In [ ]:
# 8. 下载结果摘要
# 如果仍有待重试股票，重新运行上面对应的下载单元。
summary = print_download_summary()
summary

## 9. 附录 A 可选数据下载

下面两份数据当前不进入第一版 Barra 惩罚。这里只下载和保存上游原始字段，不计算 Value、Earnings Yield、Growth、Leverage、Profitability 或 Dividend Yield。

核心财务指标必须保留 `notice_date`，后续只能在公告后使用；分红数据必须保留 `ex_dividend_date`，后续按除权除息日构造过去 12 个月分红。空响应不会标记为完成，因此下次运行仍会重试。

In [ ]:
# 10. 附录 A 股票级断点下载辅助函数（本单元不调用外部数据接口）
import shutil
import time
from collections.abc import Callable

import duckdb
import pandas as pd
import pyarrow.parquet as pq
from tqdm.auto import tqdm

APPENDIX_PART_SIZE = 50
APPENDIX_MAX_RETRIES = 3
APPENDIX_MAX_CONSECUTIVE_FAILURES = 50
APPENDIX_PARTS_ROOT = project_root / 'data' / 'download_parts'
APPENDIX_RAW_ROOT = project_root / 'data' / 'raw'
CORE_INDEX_PATH = APPENDIX_RAW_ROOT / 'stock_core_index.parquet'
DIVIDEND_PATH = APPENDIX_RAW_ROOT / 'stock_dividend.parquet'

CORE_INDEX_COLUMNS = [
    'stock_code', 'short_name', 'report_date', 'report_type', 'notice_date',
    'basic_eps', 'diluted_eps', 'non_gaap_eps', 'net_asset_ps',
    'cap_reserve_ps', 'undist_profit_ps', 'oper_cf_ps', 'total_rev',
    'gross_profit', 'net_profit_attr_sh', 'non_gaap_net_profit',
    'total_rev_yoy_gr', 'net_profit_yoy_gr', 'non_gaap_net_profit_yoy_gr',
    'total_rev_qoq_gr', 'net_profit_qoq_gr', 'non_gaap_net_profit_qoq_gr',
    'roe_wtd', 'roe_non_gaap_wtd', 'roa_wtd', 'gross_margin', 'net_margin',
    'adv_receipts_to_rev', 'net_cf_sales_to_rev', 'oper_cf_to_rev',
    'eff_tax_rate', 'curr_ratio', 'quick_ratio', 'cash_flow_ratio',
    'asset_liab_ratio', 'equity_multiplier', 'equity_ratio',
    'total_asset_turn_days', 'inv_turn_days', 'acct_recv_turn_days',
    'total_asset_turn_rate', 'inv_turn_rate', 'acct_recv_turn_rate',
]
DIVIDEND_COLUMNS = [
    'stock_code', 'report_date', 'dividend_plan', 'ex_dividend_date',
]

class AppendixEmptyResponse(RuntimeError):
    pass

def _appendix_normalize_code(values: pd.Series) -> pd.Series:
    return (
        values.astype('string').str.strip().str.replace(r'\.0$', '', regex=True).str.zfill(6)
    )

def _appendix_atomic_to_parquet(frame: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    try:
        frame.to_parquet(temporary, index=False)
        temporary.replace(path)
    finally:
        temporary.unlink(missing_ok=True)

def _appendix_part_files(dataset_name: str) -> list[Path]:
    return sorted((APPENDIX_PARTS_ROOT / dataset_name).glob('part_*.parquet'))

def _appendix_next_part(files: list[Path]) -> int:
    if not files:
        return 1
    return max(int(path.stem.removeprefix('part_')) for path in files) + 1

def _appendix_codes_in_parquet(path: Path) -> set[str]:
    parquet = pq.ParquetFile(path)
    codes: set[str] = set()
    for row_group in range(parquet.num_row_groups):
        values = parquet.read_row_group(row_group, columns=['stock_code'])['stock_code'].to_pandas()
        codes.update(_appendix_normalize_code(pd.Series(values)).dropna().astype(str))
    return codes

def _appendix_retry(request: Callable[[], pd.DataFrame], label: str) -> pd.DataFrame:
    errors: list[str] = []
    only_empty = True
    for attempt in range(1, APPENDIX_MAX_RETRIES + 1):
        try:
            response = request()
            if response is None or response.empty:
                errors.append(f'第 {attempt} 次：接口返回空数据')
            else:
                return response
        except Exception as exc:
            only_empty = False
            errors.append(f'第 {attempt} 次：{exc}')
        if attempt < APPENDIX_MAX_RETRIES:
            time.sleep(attempt)
    message = f'{label}失败；' + '；'.join(errors)
    if only_empty:
        raise AppendixEmptyResponse(message)
    raise RuntimeError(message)

def _prepare_core_index_response(response: pd.DataFrame, stock_code: str) -> pd.DataFrame:
    missing = set(CORE_INDEX_COLUMNS).difference(response.columns)
    if missing:
        raise ValueError(f'核心指标接口缺少字段：{sorted(missing)}')
    result = response.loc[:, CORE_INDEX_COLUMNS].copy()
    result['stock_code'] = str(stock_code).zfill(6)
    result['short_name'] = result['short_name'].astype('string').str.strip()
    result['report_type'] = result['report_type'].astype('string').str.strip()
    for column in ('report_date', 'notice_date'):
        result[column] = pd.to_datetime(result[column], errors='coerce').dt.normalize()
    numeric_columns = CORE_INDEX_COLUMNS[5:]
    for column in numeric_columns:
        result[column] = pd.to_numeric(result[column], errors='coerce')
    result = result.dropna(subset=['report_date', 'report_type']).copy()
    duplicate = result.duplicated(['stock_code', 'report_date', 'report_type'], keep=False)
    if duplicate.any():
        conflicts = (
            result.loc[duplicate]
            .groupby(['stock_code', 'report_date', 'report_type'], dropna=False)
            .nunique(dropna=False)
            .gt(1)
            .any(axis=1)
        )
        if conflicts.any():
            raise ValueError(f'核心指标存在同报告期冲突，示例：{conflicts.index[conflicts].tolist()[:5]}')
        result = result.drop_duplicates(['stock_code', 'report_date', 'report_type'])
    if result.empty:
        raise AppendixEmptyResponse(f'{stock_code} 核心指标清洗后为空')
    return result.sort_values(['stock_code', 'report_date', 'report_type']).reset_index(drop=True)

def _prepare_dividend_response(response: pd.DataFrame, stock_code: str) -> pd.DataFrame:
    missing = set(DIVIDEND_COLUMNS).difference(response.columns)
    if missing:
        raise ValueError(f'分红接口缺少字段：{sorted(missing)}')
    result = response.loc[:, DIVIDEND_COLUMNS].copy()
    result['stock_code'] = str(stock_code).zfill(6)
    result['dividend_plan'] = result['dividend_plan'].astype('string').str.strip()
    for column in ('report_date', 'ex_dividend_date'):
        result[column] = pd.to_datetime(result[column], errors='coerce').dt.normalize()
    result = result.dropna(subset=['report_date', 'dividend_plan']).copy()
    result = result.drop_duplicates(DIVIDEND_COLUMNS)
    if result.empty:
        raise AppendixEmptyResponse(f'{stock_code} 分红记录清洗后为空')
    return result.sort_values(['stock_code', 'report_date', 'ex_dividend_date']).reset_index(drop=True)

def _appendix_save_part(dataset_name: str, frames: list[pd.DataFrame], number: int) -> Path:
    path = APPENDIX_PARTS_ROOT / dataset_name / f'part_{number:04d}.parquet'
    _appendix_atomic_to_parquet(pd.concat(frames, ignore_index=True), path)
    return path

def _appendix_sql_path(path: Path) -> str:
    return str(path.resolve()).replace('\\', '/').replace("'", "''")

def _appendix_consolidate(
    sources: list[Path], output_path: Path, columns: list[str], key_columns: tuple[str, ...]
) -> None:
    source_sql = '[' + ','.join(f"'{_appendix_sql_path(path)}'" for path in sources) + ']'
    key_sql = ', '.join(f'"{column}"' for column in key_columns)
    value_columns = [column for column in columns if column not in key_columns]
    connection = duckdb.connect()
    temporary = output_path.with_suffix(output_path.suffix + '.tmp')
    try:
        if value_columns:
            hash_values = ', '.join(f'"{column}"' for column in value_columns)
            conflicts = connection.execute(
                f'SELECT count(*) FROM ('
                f'SELECT {key_sql} FROM read_parquet({source_sql}, union_by_name=true) '
                f'GROUP BY {key_sql} HAVING count(DISTINCT hash({hash_values})) > 1)'
            ).fetchone()[0]
            if conflicts:
                raise ValueError(f'分段中存在 {conflicts:,} 个同键冲突记录')
        output_path.parent.mkdir(parents=True, exist_ok=True)
        temporary.unlink(missing_ok=True)
        selected = ', '.join(f'"{column}"' for column in columns)
        connection.execute(
            f"COPY (SELECT {selected} FROM read_parquet({source_sql}, union_by_name=true) "
            f"QUALIFY row_number() OVER (PARTITION BY {key_sql} ORDER BY {key_sql}) = 1 "
            f"ORDER BY {key_sql}) TO '{_appendix_sql_path(temporary)}' "
            '(FORMAT PARQUET, COMPRESSION ZSTD, ROW_GROUP_SIZE 500000)'
        )
    finally:
        connection.close()
    temporary.replace(output_path)

def download_appendix_stock_dataset(
    *, dataset_name: str, output_path: Path, columns: list[str],
    key_columns: tuple[str, ...], request: Callable[[str], pd.DataFrame],
    prepare: Callable[[pd.DataFrame, str], pd.DataFrame], force_update: bool = False,
) -> dict:
    stocks = download_stock_list(force_update=False)
    codes = stocks['stock_code'].astype(str).tolist()
    parts_directory = APPENDIX_PARTS_ROOT / dataset_name
    if force_update:
        if parts_directory.parent.resolve() != APPENDIX_PARTS_ROOT.resolve():
            raise RuntimeError('断点目录越出预期范围，拒绝删除')
        shutil.rmtree(parts_directory, ignore_errors=True)
        output_path.unlink(missing_ok=True)
    files = _appendix_part_files(dataset_name)
    completed: set[str] = set()
    for path in ([output_path] if output_path.exists() else []) + files:
        completed.update(_appendix_codes_in_parquet(path))
    pending = [code for code in codes if code not in completed]
    print(f'{dataset_name}：股票池 {len(codes):,} 只，已完成 {len(completed):,} 只，待下载 {len(pending):,} 只。')
    frames: list[pd.DataFrame] = []
    failures: list[tuple[str, str]] = []
    empty_responses: list[tuple[str, str]] = []
    next_part = _appendix_next_part(files)
    consecutive_failures = 0
    for code in tqdm(pending, desc=f'下载 {dataset_name}'):
        try:
            response = _appendix_retry(lambda code=code: request(code), f'{code} {dataset_name}')
            prepared = prepare(response, code)
            frames.append(prepared)
            consecutive_failures = 0
        except AppendixEmptyResponse as exc:
            empty_responses.append((code, str(exc)))
            consecutive_failures = 0
        except Exception as exc:
            failures.append((code, str(exc)))
            consecutive_failures += 1
            if consecutive_failures >= APPENDIX_MAX_CONSECUTIVE_FAILURES:
                print('连续接口异常达到上限，停止本轮；重新运行本单元可从分片续传。')
                break
        if len(frames) >= APPENDIX_PART_SIZE:
            _appendix_save_part(dataset_name, frames, next_part)
            next_part += 1
            frames.clear()
    if frames:
        _appendix_save_part(dataset_name, frames, next_part)
        frames.clear()
    sources = ([output_path] if output_path.exists() else []) + _appendix_part_files(dataset_name)
    if sources:
        _appendix_consolidate(sources, output_path, columns, key_columns)
    final_codes = _appendix_codes_in_parquet(output_path) if output_path.exists() else set()
    missing_codes = [code for code in codes if code not in final_codes]
    print(f'最终覆盖 {len(final_codes):,}/{len(codes):,} 只；仍待复查 {len(missing_codes):,} 只。')
    if output_path.exists():
        print('已保存：', output_path)
    if empty_responses:
        print('空响应示例（未标记完成，下次仍会重试）：', empty_responses[:20])
    if failures:
        print('接口或字段失败示例：', failures[:20])
    return {
        'path': output_path, 'stock_count': len(final_codes),
        'missing_codes': missing_codes, 'empty_responses': empty_responses,
        'failures': failures,
    }

In [ ]:
# 11. 断点下载核心财务指标：get_core_index()
# 保留全部原始指标以及 report_date、notice_date；不在下载阶段做点时填充。
core_index_result = download_appendix_stock_dataset(
    dataset_name='stock_core_index',
    output_path=CORE_INDEX_PATH,
    columns=CORE_INDEX_COLUMNS,
    key_columns=('stock_code', 'report_date', 'report_type'),
    request=lambda code: adata.stock.finance.get_core_index(stock_code=code),
    prepare=_prepare_core_index_response,
    force_update=False,
)
core_index_result

In [ ]:
# 12. 断点下载历史分红：get_dividend()
# 原样保留 dividend_plan；本阶段不解析每股分红，不计算 Dividend Yield。
dividend_result = download_appendix_stock_dataset(
    dataset_name='stock_dividend',
    output_path=DIVIDEND_PATH,
    columns=DIVIDEND_COLUMNS,
    key_columns=('stock_code', 'report_date', 'dividend_plan', 'ex_dividend_date'),
    request=lambda code: adata.stock.market.get_dividend(stock_code=code),
    prepare=_prepare_dividend_response,
    force_update=False,
)
dividend_result